# Module 15 Lab — Governance Control Plane Architecture

Build a vendor-neutral runtime governance control plane around an enterprise procurement agent.

In [ ]:
%pip install -q "pydantic>=2" pandas
print("Core dependencies installed.")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional, Any
from datetime import datetime, timezone, timedelta
import hashlib, json, uuid, pandas as pd
def now(): return datetime.now(timezone.utc)
def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(",",":")).encode()).hexdigest()

## 1. Agent registry

In [ ]:
AGENTS={
 "procurement:v14":{"owner":"procurement-platform","risk":"HIGH","max_autonomy":"BOUNDED",
                    "tools":{"vendor.read","po.prepare","po.create"},"policy":"procurement-v7"},
 "research:v3":{"owner":"ai-platform","risk":"LOW","max_autonomy":"ASSISTED",
                "tools":{"vendor.read"},"policy":"research-v2"}
}
pd.DataFrame(AGENTS).T

## 2. Tool registry

In [ ]:
TOOLS={
 "vendor.read":{"risk":"LOW","permission":"vendor.read","reversible":True},
 "po.create":{"risk":"HIGH","permission":"po.create","reversible":True},
 "payment.execute":{"risk":"CRITICAL","permission":"payment.execute","reversible":False}
}
pd.DataFrame(TOOLS).T

## 3. Delegated authority

In [ ]:
class Delegation(BaseModel):
    id:str=Field(default_factory=lambda:"del-"+uuid.uuid4().hex[:10])
    delegator:str
    delegate:str
    purpose:str
    permissions:set[str]
    constraints:dict[str,Any]={}
    expires_at:datetime

delegation=Delegation(delegator="user:42",delegate="procurement:v14",
 purpose="purchase approved equipment",permissions={"vendor.read","po.create"},
 constraints={"max_amount":15000},expires_at=now()+timedelta(hours=1))
delegation

## 4. Authority attenuation

In [ ]:
def delegate(parent:Delegation,child_agent:str,permissions:set[str],constraints:dict):
    if not permissions.issubset(parent.permissions):
        raise PermissionError("Authority amplification blocked")
    parent_max=parent.constraints.get("max_amount",float("inf"))
    child_max=constraints.get("max_amount",parent_max)
    if child_max>parent_max:
        raise PermissionError("Constraint widening blocked")
    return Delegation(delegator=parent.delegate,delegate=child_agent,purpose=parent.purpose,
                      permissions=permissions,constraints={**constraints,"max_amount":child_max},
                      expires_at=min(parent.expires_at,now()+timedelta(minutes=30)))
child=delegate(delegation,"research:v3",{"vendor.read"},{"max_amount":0})
child

## 5. Governance context envelope

In [ ]:
class ActionRequest(BaseModel):
    request_id:str=Field(default_factory=lambda:"req-"+uuid.uuid4().hex[:10])
    principal:str
    agent:str
    delegation_id:str
    purpose:str
    tool:str
    arguments:dict[str,Any]
    data_classification:str="INTERNAL"
    environment:str="prod"

req=ActionRequest(principal="user:42",agent="procurement:v14",delegation_id=delegation.id,
 purpose=delegation.purpose,tool="po.create",arguments={"vendor":"V42","amount":12000})
req

## 6. Contextual risk engine

In [ ]:
def risk_score(req):
    score=0.1
    amount=req.arguments.get("amount",0)
    if amount>=10000: score+=.35
    if amount>=25000: score+=.25
    if req.data_classification=="RESTRICTED": score+=.25
    if not TOOLS[req.tool]["reversible"]: score+=.25
    return min(score,1.0)
risk_score(req)

## 7. Policy decision

In [ ]:
class Decision(BaseModel):
    id:str=Field(default_factory=lambda:"dec-"+uuid.uuid4().hex[:10])
    result:Literal["ALLOW","DENY","ESCALATE","CONSTRAIN"]
    policy_version:str
    reason_codes:list[str]
    risk_score:float

def decide(req:ActionRequest,d:Delegation):
    reasons=[]; tool=TOOLS.get(req.tool)
    if not tool:return Decision(result="DENY",policy_version="procurement-v7",reason_codes=["UNKNOWN_TOOL"],risk_score=1)
    if req.agent!=d.delegate or now()>d.expires_at: reasons.append("INVALID_DELEGATION")
    if tool["permission"] not in d.permissions: reasons.append("NO_PERMISSION")
    amount=req.arguments.get("amount",0)
    if amount>d.constraints.get("max_amount",float("inf")): reasons.append("DELEGATION_LIMIT")
    r=risk_score(req)
    if reasons:return Decision(result="DENY",policy_version="procurement-v7",reason_codes=reasons,risk_score=r)
    if r>=.7:return Decision(result="ESCALATE",policy_version="procurement-v7",reason_codes=["HIGH_RISK"],risk_score=r)
    if amount>=10000:return Decision(result="ESCALATE",policy_version="procurement-v7",reason_codes=["HIGH_VALUE"],risk_score=r)
    return Decision(result="ALLOW",policy_version="procurement-v7",reason_codes=["WITHIN_POLICY"],risk_score=r)
decision=decide(req,delegation); decision

## 8. Action-bound approval

In [ ]:
def action_fingerprint(req): return digest({"tool":req.tool,"arguments":req.arguments,"principal":req.principal})
approval={"approval_id":"app-901","action_hash":action_fingerprint(req),"approver":"manager:7",
          "expires_at":(now()+timedelta(minutes=10)).isoformat()}
def approval_valid(req,approval):
    return approval["action_hash"]==action_fingerprint(req) and datetime.fromisoformat(approval["expires_at"])>now()
approval_valid(req,approval)

## 9. Enforcement point / tool gateway

In [ ]:
EXECUTED=[]
def execute(req,d,approval=None):
    dec=decide(req,d)
    if dec.result=="DENY": return {"status":"BLOCKED","decision":dec}
    if dec.result=="ESCALATE" and not (approval and approval_valid(req,approval)):
        return {"status":"AWAITING_APPROVAL","decision":dec}
    EXECUTED.append({"tool":req.tool,"args":req.arguments})
    return {"status":"EXECUTED","decision":dec,"transaction":"tx-"+uuid.uuid4().hex[:8]}
execute(req,delegation,approval)

## 10. Approval mutation attack

In [ ]:
mutated=req.model_copy(deep=True); mutated.arguments["amount"]=24000
print("approval valid:",approval_valid(mutated,approval))
execute(mutated,delegation,approval)["status"]

## 11. CONSTRAIN decisions

In [ ]:
def data_export_policy(classification,destination):
    if classification=="RESTRICTED" and destination=="external":
        return {"decision":"CONSTRAIN","obligations":["REDACT_PII","MAX_100_ROWS"]}
    return {"decision":"ALLOW","obligations":[]}
data_export_policy("RESTRICTED","external")

## 12. MCP-like tool discovery filtering

In [ ]:
def discover_tools(agent):
    allowed=AGENTS[agent]["tools"]
    return {name:meta for name,meta in TOOLS.items() if name in allowed}
discover_tools("research:v3")

## 13. Multi-agent propagation

In [ ]:
envelope={
 "caller":"procurement:v14","callee":"research:v3","delegation_id":child.id,
 "purpose":child.purpose,"permissions":sorted(child.permissions),
 "trace_id":"trace-"+uuid.uuid4().hex[:12]
}
envelope

## 14. Decision evidence

In [ ]:
def evidence(req,dec):
    return {"decision_id":dec.id,"request_id":req.request_id,"agent":req.agent,
            "principal":req.principal,"tool":req.tool,"policy":dec.policy_version,
            "decision":dec.result,"reason_codes":dec.reason_codes,"risk":dec.risk_score,
            "action_hash":action_fingerprint(req)}
evidence(req,decision)

## 15. Safe decision cache

In [ ]:
CACHE={}
def cache_key(req,d,policy_version):
    return digest({"agent":req.agent,"principal":req.principal,"tool":req.tool,
                   "args":req.arguments,"delegation":d.id,"policy":policy_version})
k=cache_key(req,delegation,"procurement-v7")
CACHE[k]=decision.model_dump()
len(CACHE)

## 16. Fail-closed behavior

In [ ]:
def governed_call(policy_available,risk_tier):
    if not policy_available:
        if risk_tier in {"HIGH","CRITICAL"}: return "DENY_POLICY_UNAVAILABLE"
        return "DEGRADED_READ_ONLY"
    return "EVALUATE"
[governed_call(False,x) for x in ["LOW","HIGH","CRITICAL"]]

## 17. Shadow policy

In [ ]:
def policy_v8(req,d):
    old=decide(req,d)
    if req.tool=="po.create" and req.arguments.get("amount",0)>=5000 and old.result=="ALLOW":
        return old.model_copy(update={"result":"ESCALATE","policy_version":"procurement-v8","reason_codes":["V8_LOWER_THRESHOLD"]})
    return old.model_copy(update={"policy_version":"procurement-v8"})
test=req.model_copy(deep=True); test.arguments["amount"]=7000
print("current:",decide(test,delegation).result,"candidate:",policy_v8(test,delegation).result)

## 18. OPA/Rego policy example

In [ ]:
rego = r"""
package agent.governance

default decision := {"result": "DENY", "reason": "default_deny"}

decision := {"result": "ALLOW", "reason": "authorized"} if {
  input.tool == "vendor.read"
  "vendor.read" in input.permissions
}

decision := {"result": "ESCALATE", "reason": "high_value"} if {
  input.tool == "po.create"
  "po.create" in input.permissions
  input.amount >= 10000
}
"""
print(rego)

## 19. OPA integration pattern

In production, the enforcement point can send structured input to an OPA decision endpoint:

```text
Tool Gateway
   ↓
POST policy input
   ↓
OPA / Rego
   ↓
structured decision
   ↓
Gateway enforces
```

Secure the policy service itself with TLS, authentication and authorization. OPA's own documentation notes these protections are not automatically enabled in all deployment modes.

## 20. OpenTelemetry governance metadata

In [ ]:
otel_attributes={
 "governance.agent.id":req.agent,
 "governance.decision.id":decision.id,
 "governance.decision.result":decision.result,
 "governance.policy.version":decision.policy_version,
 "governance.risk.score":decision.risk_score,
 "governance.delegation.id":delegation.id
}
otel_attributes

## 21. Correctness invariants

In [ ]:
assert req.agent==delegation.delegate
assert set(child.permissions).issubset(delegation.permissions)
assert child.constraints["max_amount"]<=delegation.constraints["max_amount"]
assert approval_valid(req,approval)
assert not approval_valid(mutated,approval)
print("Core governance invariants hold.")

## 22. CI governance tests

In [ ]:
tests={
 "unregistered_tool_denied": decide(req.model_copy(update={"tool":"shell.exec"}),delegation).result=="DENY",
 "permission_required": decide(req.model_copy(update={"tool":"payment.execute"}),delegation).result=="DENY",
 "high_value_escalates": decide(req,delegation).result=="ESCALATE",
 "mutated_approval_invalid": not approval_valid(mutated,approval),
}
assert all(tests.values()),tests
tests

## 23. Exercises

1. Add an Agent Registry API and lifecycle states.
2. Add owner, environment and version constraints.
3. Implement a signed delegation token.
4. Create a three-agent authority chain and prove attenuation.
5. Add ABAC rules for data classification.
6. Add a policy decision point using a local OPA server.
7. Add policy bundle versioning.
8. Implement ALLOW/DENY/ESCALATE/CONSTRAIN obligations.
9. Add two-person approval for irreversible transactions.
10. Build an MCP tool gateway with filtered discovery.
11. Add short-lived execution credentials.
12. Add risk-aware cache TTLs.
13. Simulate policy-engine failure.
14. Shadow-test a candidate policy over 100 requests.
15. Export decision evidence through OpenTelemetry.
16. Feed a Module 14 evaluation failure into a policy update.
17. Add an incident kill switch for one agent/tool.
18. Threat-model the control plane itself.

## 24. Key takeaways

- Separate reasoning from authority.
- Register agents and tools.
- Model delegated authority explicitly.
- Prevent authority amplification.
- Normalize governance context.
- Externalize and version policy.
- Enforce at consequential boundaries.
- Use richer decisions than allow/deny.
- Bind approvals to exact actions.
- Govern MCP and agent-to-agent calls.
- Integrate enterprise data classification.
- Prefer scoped, short-lived execution credentials.
- Design fail-closed/degraded behavior by risk.
- Avoid a single synchronous mega-gateway.
- Record decision evidence.
- Feed observability and evaluation back into controls.
- The control plane is an architecture, not necessarily one product.